<a href="https://colab.research.google.com/github/PRIYADARSHINI12-tech/Cross-market-Analysis/blob/main/CROSS_MARKET_ANALYSIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests

DATA COLLECTION

In [ ]:
import requests

url1 = "https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&page=1&per_page=250"

response1 = requests.get(url1)

data_page1 = response1.json()

print(len(data_page1))

250


In [ ]:
url2 = "https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&page=2&per_page=250"

response2 = requests.get(url2)

data_page2 = response2.json()

print(len(data_page2))

250


In [ ]:
url3 = "https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&page=3&per_page=250"

response3 = requests.get(url3)

data_page3 = response3.json()

print(len(data_page3))

250


In [ ]:
url4 = "https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&page=4&per_page=250"

response4 = requests.get(url4)

data_page4 = response4.json()

print(len(data_page4))

1


In [ ]:
url5 = "https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&page=5&per_page=250"

response5 = requests.get(url5)

data_page5 = response5.json()

print(len(data_page5))

1


In [ ]:
all_records =[]
all_records.extend(data_page1)
all_records.extend(data_page2)
all_records.extend(data_page3)
all_records.extend(data_page4)
all_records.extend(data_page5)

print(len(all_records))

752


In [ ]:
import pandas as pd
df = pd.DataFrame(all_records)
df.head()

AttributeError: 'str' object has no attribute 'keys'

FILTERING THE COLUMNS

In [ ]:
df_filtered = df[['id',
                  'symbol',
                  'name',
                  'current_price',
                  'market_cap',
                  'market_cap_rank',
                  'total_volume',
                  'circulating_supply',
                  'total_supply',
                  'ath',
                  'atl',
                  'last_updated']].copy()
df_filtered.rename(columns={'last_updated':'Date'}, inplace=True)

df_filtered['Date'] = pd.to_datetime(df_filtered['Date']).dt.date


df_filtered.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df_filtered.to_csv("/content/drive/MyDrive/crypto_filtered_data.csv", index=False)

df_filtered.to_csv("crypto_data.csv", index=False)

CRYPTO PRICES

In [ ]:
import requests
import pandas as pd
import time

coins = ['bitcoin', 'ethereum', 'tether']
all_data = []

for coin_id in coins:
    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart?vs_currency=usd&days=365"

    response = requests.get(url)
    print(f"{coin_id}: {response.status_code}")

    time.sleep(2)

    if response.status_code == 200:
        data = response.json()
        prices = data['prices']

        df = pd.DataFrame(prices, columns=['timestamp', 'price_usd'])
        df['date'] = pd.to_datetime(df['timestamp'], unit='ms').dt.date
        df['coin_id'] = coin_id

        df = df[['coin_id', 'date', 'price_usd']]
        all_data.append(df)
    else:
        print(f"Error for {coin_id}: {response.status_code}")

# Combine all data
final_df = pd.concat(all_data)

# OUTPUT
print(final_df.head())
print(final_df['coin_id'].unique())

In [ ]:
final_df.to_csv("crypto_prices.csv", index=False)
print(final_df['coin_id'].unique())

OIL PRICES

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/datasets/oil-prices/main/data/wti-daily.csv"

oil_df = pd.read_csv(url)

print(oil_df.head())

In [ ]:
filtered_oil = oil_df[
    (oil_df['Date'] >= '2020-01-01') &
    (oil_df['Date'] <= '2026-03-23')
]
print(filtered_oil.head())
filtered_oil.to_csv("oil_prices.csv", index=False)


STOCK PRICES

In [ ]:
import yfinance as yf
import pandas as pd

tickers = ["^GSPC", "^IXIC", "^NSEI"]
start_date = "2020-01-01"
end_date = "2026-03-23"

data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker')

dfs = []

for ticker in tickers:
    df = data[ticker].reset_index()
    df['Ticker'] = ticker
    dfs.append(df)

stock_df = pd.concat(dfs, ignore_index=True)

# Check properly
print(stock_df.head())
print(stock_df['Ticker'].unique())

In [ ]:
print(data.isna().sum())

In [ ]:
data = data.ffill().bfill()

In [ ]:
print(data.isna().sum())

In [ ]:
stock_df.to_csv("stock_prices.csv", index=False)

CREATING DATABASE

In [ ]:
import sqlite3
conn = sqlite3.connect("MARKET.db")
cursor = conn.cursor()

In [ ]:
df_filtered.to_sql("crypto_data",if_exists="replace",con = conn)
pd.read_sql("select * from crypto_data",con = conn)

In [ ]:
final_df.to_sql("crypto_prices",if_exists="replace",con = conn)
pd.read_sql("select * from crypto_prices",con = conn)

In [ ]:
oil_df.to_sql("oil_prices",if_exists="replace",con = conn)
pd.read_sql("select * from oil_prices",con = conn)


In [ ]:
stock_df.to_sql("stock_prices",if_exists="replace",con = conn)
pd.read_sql("select * from stock_prices",con = conn)

In [ ]:
stock_df['Date'] = pd.to_datetime(stock_df['Date']).dt.date

In [ ]:
stock_df.to_sql("stock_prices", conn, if_exists="replace", index=False)

In [ ]:
pd.read_sql("SELECT Date FROM stock_prices LIMIT 5;", conn)

CRYPTOCURRENCIES TABLE

In [ ]:
import sqlite3

conn = sqlite3.connect("MARKET.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS cryptocurrencies (
    id TEXT PRIMARY KEY,
    symbol TEXT,
    name TEXT,
    current_price REAL,
    total_volume INTEGER,
    circulating_supply REAL,
    total_supply REAL,
    ath REAL,
    atl REAL,
    date DATE
);
""")

conn.commit()

In [ ]:
df_filtered.to_sql("crypto_data",if_exists="replace",con = conn)
pd.read_sql("select * from crypto_data",con = conn)

In [ ]:
import pandas as pd

pd.read_sql("SELECT * FROM cryptocurrencies LIMIT 5;", conn)


In [ ]:
df_filtered.to_sql("cryptocurrencies", conn, if_exists="replace", index=False)

CRYPTO PRICES TABLE

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS crypto_prices (
    coin_id TEXT,
    date TEXT,
    price_usd REAL,
    PRIMARY KEY (coin_id, date),
    FOREIGN KEY (coin_id) REFERENCES cryptocurrencies(id)
);
""")

conn.commit()

In [ ]:
import pandas as pd

df = pd.read_csv("crypto_prices.csv")

final_df.to_sql("crypto_prices", conn, if_exists="append", index=False)

OIL PRICES TABLE

In [ ]:
import sqlite3

conn = sqlite3.connect("MARKET.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS oil_prices (
    date TEXT PRIMARY KEY,
    price_usd REAL
);
""")

conn.commit()

In [ ]:
import pandas as pd

pd.read_sql("SELECT * FROM oil_prices LIMIT 5;", conn)

STOCK PRICES TABLE

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS stock_prices (
    date TEXT,
    open REAL,
    high REAL,
    low REAL,
    close REAL,
    volume INTEGER,
    ticker TEXT,
    PRIMARY KEY (date, ticker)
);
""")

conn.commit()

In [ ]:
import pandas as pd

pd.read_sql("SELECT * FROM stock_prices LIMIT 5;", conn)

QUERIES

1.Find the top 3 cryptocurrencies by market cap.

In [ ]:
cursor.execute("""
SELECT name, market_cap
FROM cryptocurrencies
ORDER BY market_cap DESC
LIMIT 3;
""")

print(cursor.fetchall())

2.List all coins where circulating supply exceeds 90% of total supply.

In [ ]:
cursor.execute("""
SELECT name,
       circulating_supply,
       total_supply,
       (circulating_supply * 100.0 / total_supply) AS percentage
FROM cryptocurrencies
WHERE circulating_supply > 0.9 * total_supply
AND circulating_supply < total_supply;
""")

rows=cursor.fetchall()
print(rows)

for name, circ, total, pct in rows:
    print(f"{name}: {pct:.2f}% circulating")

3.Get coins that are within 10% of their all-time-high (ATH).



In [ ]:
cursor.execute("""
SELECT name, current_price, ath
FROM cryptocurrencies
WHERE current_price >= 0.9 * ath;
""")
rows=cursor.fetchall()
print(rows)


4.Find the average market cap rank of coins with volume above $1B.



In [ ]:
cursor.execute("""
SELECT AVG(market_cap_rank)
FROM cryptocurrencies
WHERE total_volume > 1000000000;
""")

result = cursor.fetchone()
print(result)

5.Get the most recently updated coin.

In [ ]:
cursor.execute("""
SELECT name, date
FROM cryptocurrencies
ORDER BY date DESC
LIMIT 1;
""")

print(cursor.fetchone())

2️⃣ crypto_prices (daily prices of top coins)

1.Find the highest daily price of Bitcoin in the last 365 days.



In [ ]:
cursor.execute("""
SELECT MAX(price_usd)
FROM crypto_prices
WHERE coin_id = 'bitcoin'
AND date >= DATE('now', '-365 days');
""")

print(cursor.fetchone())

2.Calculate the average daily price of Ethereum in the past 1 year.

In [ ]:
cursor.execute("""
SELECT AVG(price_usd)
FROM crypto_prices
WHERE coin_id = 'ethereum'
AND date >= DATE('now', '-1 year');
""")

print(cursor.fetchone())

3.Show the daily price trend of Bitcoin in January 2026.



In [ ]:
cursor.execute("""
SELECT DISTINCT date, price_usd
FROM crypto_prices
WHERE coin_id = 'bitcoin'
AND strftime('%Y-%m', date) = '2026-01'
ORDER BY date;
""")

print(cursor.fetchall())

[('2026-01-01', 87520.18048742875), ('2026-01-02', 88727.67004249741), ('2026-01-03', 89926.27924953419), ('2026-01-04', 90593.85443180417), ('2026-01-05', 91373.21588108438), ('2026-01-06', 93926.7956485658), ('2026-01-07', 93666.86387314305), ('2026-01-08', 91257.15544035907), ('2026-01-09', 90983.51578142625), ('2026-01-10', 90504.89560340019), ('2026-01-11', 90442.01907847001), ('2026-01-12', 90819.36598904192), ('2026-01-13', 91134.96917951507), ('2026-01-14', 95260.4422971001), ('2026-01-15', 97007.78078881276), ('2026-01-16', 95584.82797878512), ('2026-01-17', 95516.07771987123), ('2026-01-18', 95099.53192554631), ('2026-01-19', 93752.70577188236), ('2026-01-20', 92558.46334372147), ('2026-01-21', 88312.84053255555), ('2026-01-22', 89354.34377512129), ('2026-01-23', 89443.39744146909), ('2026-01-24', 89412.39849953113), ('2026-01-25', 89170.87364531498), ('2026-01-26', 86548.32213469829), ('2026-01-27', 88307.8612070438), ('2026-01-28', 89204.22239644526), ('2026-01-29', 89162.0

4.Find the coin with the highest average price over 1 year.

In [ ]:
cursor.execute("""
SELECT coin_id, AVG(price_usd) AS avg_price
FROM crypto_prices
WHERE date >= DATE('now', '-1 year')
GROUP BY coin_id
ORDER BY avg_price DESC
LIMIT 1;
""")

print(cursor.fetchone())

5.Get the % change in Bitcoin’s price between Mar 2025 and Mar 2026.

In [ ]:
cursor.execute("""
SELECT
(
    (SELECT price_usd FROM crypto_prices
     WHERE coin_id='bitcoin'
     AND strftime('%Y-%m', date) = '2026-03'
     ORDER BY date DESC LIMIT 1)

    -

    (SELECT price_usd FROM crypto_prices
     WHERE coin_id='bitcoin'
     AND strftime('%Y-%m', date) = '2025-03'
     ORDER BY date DESC LIMIT 1)
)
* 100.0 /

(SELECT price_usd FROM crypto_prices
 WHERE coin_id='bitcoin'
 AND strftime('%Y-%m', date) = '2025-03'
 ORDER BY date DESC
 LIMIT 1)
 AS percent_change;
""")
result = cursor.fetchone()
print("Percentage Change:",result[0])


OIL PRICES 1.Find the highest oil price in the last 5 years.





In [ ]:
cursor.execute("""
SELECT MAX(price)
FROM oil_prices
WHERE date >= DATE('now', '-5 years');
""")

print(cursor.fetchone())

2.Get the average oil price per year.

In [ ]:
cursor.execute("""
SELECT strftime('%Y', date) AS year, AVG(price)
FROM oil_prices
GROUP BY year
ORDER BY year;
""")

print(cursor.fetchall())

3.Show oil prices during COVID crash (March–April 2020).

In [ ]:
cursor.execute("""
SELECT date, price
FROM oil_prices
WHERE date BETWEEN '2020-03-01' AND '2020-04-30'
ORDER BY date;
""")

print(cursor.fetchall())

[('2020-03-02', 46.78), ('2020-03-03', 47.27), ('2020-03-04', 46.78), ('2020-03-05', 45.9), ('2020-03-06', 41.14), ('2020-03-09', 31.05), ('2020-03-10', 34.47), ('2020-03-11', 33.13), ('2020-03-12', 31.56), ('2020-03-13', 31.72), ('2020-03-16', 28.96), ('2020-03-17', 26.96), ('2020-03-18', 20.48), ('2020-03-19', 25.09), ('2020-03-20', 19.48), ('2020-03-23', 23.33), ('2020-03-24', 21.03), ('2020-03-25', 20.75), ('2020-03-26', 16.6), ('2020-03-27', 15.48), ('2020-03-30', 14.1), ('2020-03-31', 20.51), ('2020-04-01', 20.28), ('2020-04-02', 25.18), ('2020-04-03', 28.36), ('2020-04-06', 26.21), ('2020-04-07', 23.54), ('2020-04-08', 24.97), ('2020-04-09', 22.9), ('2020-04-13', 22.36), ('2020-04-14', 20.15), ('2020-04-15', 19.96), ('2020-04-16', 19.82), ('2020-04-17', 18.31), ('2020-04-20', -36.98), ('2020-04-21', 8.91), ('2020-04-22', 13.64), ('2020-04-23', 15.06), ('2020-04-24', 15.99), ('2020-04-27', 12.17), ('2020-04-28', 12.4), ('2020-04-29', 15.04), ('2020-04-30', 19.23)]


4.Find the lowest price of oil in the last 10 years.

In [ ]:
cursor.execute("""
SELECT date, price
FROM oil_prices
ORDER BY price ASC
LIMIT 5;
""")

print(cursor.fetchall())

[('2020-04-20', -36.98), ('2020-04-21', 8.91), ('1986-03-31', 10.25), ('1998-12-10', 10.82), ('1986-07-23', 10.83)]


In [ ]:
cursor.execute("""
SELECT MIN(price)
FROM oil_prices
WHERE date >= DATE('now', '-10 years');
""")


print(cursor.fetchone())

(-36.98,)


5.Calculate the volatility of oil prices (max-min difference per year).




In [ ]:
cursor.execute("""
SELECT
strftime('%Y', date) AS year,
MAX(price) - MIN(price) AS volatility
FROM oil_prices
GROUP BY year
ORDER BY year;
""")

print(cursor.fetchall())

4️⃣ stock_prices

1.Get all stock prices for a given ticker

In [ ]:
cursor.execute("""
SELECT *
FROM stock_prices
WHERE ticker IN ('^GSPC', '^IXIC', '^NSEI')
ORDER BY ticker,date;
""")
for row in cursor.fetchall():
    print(row)


2.Find the highest closing price for NASDAQ (^IXIC)

In [ ]:
cursor.execute("""
SELECT MAX(close)
FROM stock_prices
WHERE ticker = '^IXIC';
""")

print(cursor.fetchone())

3.List top 5 days with highest price difference (high - low) for S&P 500 (^GSPC)

In [ ]:
cursor.execute("""
SELECT Date, High, Low, (High - Low) AS price_diff
FROM stock_prices
WHERE Ticker = '^GSPC'
ORDER BY price_diff DESC
LIMIT 5;
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

4.Get monthly average closing price for each ticker

In [ ]:
cursor.execute("""
SELECT Ticker,
       strftime('%Y-%m', Date) AS month,
       AVG(Close) AS avg_close
FROM stock_prices
GROUP BY Ticker, month
ORDER BY Ticker, month;
""")

rows = cursor.fetchall()

for row in rows:

    print(f"Ticker: {row[0]}, Month: {row[1]}, Avg Close: {row[2]:.2f}")

5.Get average trading volume of NSEI in 2026



In [ ]:
cursor.execute("""
SELECT AVG(Volume)
FROM stock_prices
WHERE Ticker = '^NSEI'
AND strftime('%Y', Date) = '2026';
""")

print(cursor.fetchone())

🔗 Join Queries (Cross-Market Analysis)

1.Compare Bitcoin vs Oil average price in 2026.



In [ ]:
cursor.execute("""
SELECT
    (SELECT AVG(current_price)
     FROM cryptocurrencies
     WHERE id='bitcoin' AND strftime('%Y', date)='2026') AS btc_avg,

    (SELECT AVG(price)
     FROM oil_prices
     WHERE strftime('%Y', date)='2026') AS oil_avg;
""")

print(cursor.fetchone())

2.Check if Bitcoin moves with S&P 500 (correlation idea).

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("MARKET.db")
cursor = conn.cursor()

query = """
SELECT
    DATE(p.date) as date,
    AVG(p.price_usd) AS bitcoin_price,
    AVG(s.Close) AS sp500
FROM crypto_prices p
INNER JOIN stock_prices s
ON DATE(p.date) = DATE(s.Date)
AND s.Ticker = '^GSPC'
WHERE p.coin_id = 'bitcoin'
AND strftime('%Y', p.date) = '2026'
GROUP BY DATE(p.date)
ORDER BY date;
"""

cursor.execute(query)
rows = cursor.fetchall()

df = pd.DataFrame(rows, columns=['date', 'bitcoin_price', 'sp500'])


df = df.dropna()


correlation = df['bitcoin_price'].corr(df['sp500'])


print("Correlation (Bitcoin vs S&P 500 - 2026):", correlation)

conn.close()

3.Compare Ethereum and NASDAQ daily prices for 2026.

In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("MARKET.db")
cursor = conn.cursor()

query = """
SELECT
    p.date,
    p.price_usd AS ethereum_price,
    s.Close AS nasdaq
FROM crypto_prices p
INNER JOIN stock_prices s
ON DATE(p.date) = DATE(s.Date)
AND s.Ticker = '^IXIC'
WHERE p.coin_id = 'ethereum'
AND strftime('%Y', p.date) = '2026'
ORDER BY p.date;
"""


cursor.execute(query)
rows = cursor.fetchall()


df = pd.DataFrame(rows, columns=['date', 'ethereum_price', 'nasdaq'])


df = df.dropna()


correlation = df['ethereum_price'].corr(df['nasdaq'])


print("Correlation (Ethereum vs NASDAQ - 2026):", correlation)

# Close connection
conn.close()

4.Find days when oil price spiked and compare with Bitcoin price change

In [ ]:
import sqlite3
import pandas as pd

# Connect
conn = sqlite3.connect("MARKET.db")
cursor = conn.cursor()

query = """
SELECT
    DATE(o.Date) as Date,
    AVG(o.Price) AS oil_price,
    p.price_usd AS bitcoin_price
FROM oil_prices o
INNER JOIN crypto_prices p
ON DATE(o.Date) = DATE(p.date)
WHERE p.coin_id = 'bitcoin'
GROUP BY DATE(o.Date), p.price_usd
ORDER BY Date;
"""

df = pd.read_sql(query, conn).dropna()

# Calculate BTC change correctly
df['btc_change'] = df['bitcoin_price'].diff()

print(df.head())


5.Compare top 3 coins daily price trend vs Nifty (^NSEI).

In [ ]:
query = """
SELECT
    p.date,
    p.coin_id,
    p.price_usd,
    s.Close AS nifty
FROM crypto_prices p
INNER JOIN stock_prices s
ON DATE(p.date) = DATE(s.Date)
WHERE p.coin_id IN ('bitcoin','ethereum','tether')
AND s.Ticker = '^NSEI'
ORDER BY p.date;
"""

df = pd.read_sql(query, conn).dropna()

print(df.head())

         date   coin_id     price_usd         nifty
0  2025-03-24   bitcoin  85787.709149  23658.349609
1  2025-03-24  ethereum   2001.047023  23658.349609
2  2025-03-24    tether      0.999964  23658.349609
3  2025-03-24   bitcoin  85787.709149  23658.349609
4  2025-03-24  ethereum   2001.047023  23658.349609


6.Compare stock prices (^GSPC) with crude oil prices on the same dates

In [ ]:
query = """
SELECT
    s.Date,
    s.Close AS sp500,
    o.Price AS oil_price
FROM stock_prices s
INNER JOIN oil_prices o
ON DATE(s.Date) = DATE(o.Date)
WHERE s.Ticker = '^GSPC'
ORDER BY s.Date;
"""

df = pd.read_sql(query, conn).dropna()

print(df.head())

7.Correlate Bitcoin closing price with crude oil closing price (same date)

In [ ]:
query = """
SELECT
    p.date,
    p.price_usd AS bitcoin,
    o.Price AS oil
FROM crypto_prices p
INNER JOIN oil_prices o
ON DATE(p.date) = DATE(o.Date)
WHERE p.coin_id = 'bitcoin'
ORDER BY p.date;
"""

df = pd.read_sql(query, conn).dropna()

correlation = df['bitcoin'].corr(df['oil'])

print("Correlation (Bitcoin vs Oil):", correlation)

8.Compare NASDAQ (^IXIC) with Ethereum price trends

In [ ]:
query = """
SELECT
    p.date,
    AVG(p.price_usd) AS ethereum,
    AVG(s.close) AS nasdaq
FROM crypto_prices p
JOIN stock_prices s
ON DATE(p.date) = DATE(s.date)
WHERE p.coin_id = 'ethereum'
AND s.ticker = '^IXIC'
GROUP BY p.date
ORDER BY p.date;
"""

df = pd.read_sql(query, conn).dropna()

print(df.head())

print("Correlation:", df['ethereum'].corr(df['nasdaq']))

         date     ethereum        nasdaq
0  2025-03-24  2001.047023  18188.589844
1  2025-03-25  2077.739326  18271.859375
2  2025-03-26  2068.598435  17899.019531
3  2025-03-27  2009.883543  17804.029297
4  2025-03-28  2003.303424  17322.990234
Correlation: 0.5078557804053746


9.Join top 3 crypto coins with stock indices for 2026

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("MARKET.db")
cursor = conn.cursor()

query = """
SELECT
    DATE(p.date) AS date,
    p.coin_id,
    AVG(p.price_usd) AS price_usd,
    MAX(CASE WHEN s.Ticker='^NSEI' THEN s.Close END) AS nsei,
    MAX(CASE WHEN s.Ticker='^GSPC' THEN s.Close END) AS sp500,
    MAX(CASE WHEN s.Ticker='^IXIC' THEN s.Close END) AS nasdaq
FROM crypto_prices p
INNER JOIN stock_prices s
ON DATE(p.date) = DATE(s.Date)
WHERE p.coin_id IN ('bitcoin','ethereum','tether')
AND strftime('%Y', p.date) = '2026'
GROUP BY DATE(p.date), p.coin_id
ORDER BY date;
"""

df = pd.read_sql(query, conn).dropna()

print(df.head())

         date   coin_id     price_usd          nsei        sp500        nasdaq
3  2026-01-02   bitcoin  88727.670042  26328.550781  6858.470215  23235.630859
4  2026-01-02  ethereum   3000.419117  26328.550781  6858.470215  23235.630859
5  2026-01-02    tether      0.998868  26328.550781  6858.470215  23235.630859
6  2026-01-05   bitcoin  91373.215881  26250.300781  6902.049805  23395.820312
7  2026-01-05  ethereum   3139.057042  26250.300781  6902.049805  23395.820312


10.Multi-join: stock prices, oil prices, and Bitcoin prices for daily comparison

In [ ]:
query = """
SELECT
    p.date,
    p.price_usd AS bitcoin,
    s.close AS sp500,
    o.price AS oil
FROM crypto_prices p
JOIN (
    SELECT date, close
    FROM stock_prices
    WHERE ticker = '^GSPC'
) s ON DATE(p.date) = DATE(s.date)
JOIN (
    SELECT date, price
    FROM oil_prices
) o ON DATE(p.date) = DATE(o.date)
WHERE p.coin_id = 'bitcoin'
GROUP BY p.date
ORDER BY p.date;
"""

df = pd.read_sql(query, conn).dropna()

print(df.head())

print("\nCorrelation Matrix:\n")
print(df[['bitcoin','sp500','oil']].corr())

         date       bitcoin        sp500    oil
0  2025-03-24  85787.709149  5767.569824  69.46
1  2025-03-25  87327.729697  5776.649902  69.48
2  2025-03-26  87520.583915  5712.200195  70.05
3  2025-03-27  86960.855549  5693.310059  70.30
4  2025-03-28  87227.271580  5580.939941  69.74

Correlation Matrix:

          bitcoin     sp500       oil
bitcoin  1.000000 -0.100210 -0.168472
sp500   -0.100210  1.000000 -0.115128
oil     -0.168472 -0.115128  1.000000


Streamlit Structure

In [ ]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

--2026-03-24 11:58:01--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64 [following]
--2026-03-24 11:58:01--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/731ab2f8-6b77-4adb-a7b3-1104525e9d72?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-24T12%3A45%3A15Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-24T1

In [ ]:
!pip install streamlit-option-menu

In [ ]:
%%writefile app.py
import streamlit as st
import sqlite3
import pandas as pd

# -------------------------------
# PAGE CONFIG
# -------------------------------
st.set_page_config(page_title="Cross Market Analysis", layout="wide")

# -------------------------------
# DATABASE CONNECTION
# -------------------------------
conn = sqlite3.connect("MARKET.db")

# -------------------------------
# SIDEBAR NAVIGATION
# -------------------------------
st.sidebar.title("📌 Navigation")
page = st.sidebar.radio(
    "Go to",
    ["Page 1: Overview", "Page 2: SQL Runner", "Page 3: Crypto Analysis"]
)

# =========================================================
# 📊 PAGE 1: OVERVIEW
# =========================================================
if page == "Page 1: Overview":

    st.title("📊 Cross-Market Overview")

    date_range = st.sidebar.date_input("Select Date Range", [])

    query = """
    SELECT
        p.date,
        p.price_usd AS bitcoin,
        o.price AS oil,
        s.Close AS sp500,
        n.Close AS nifty
    FROM crypto_prices p
    JOIN oil_prices o ON DATE(p.date)=DATE(o.date)
    JOIN stock_prices s ON DATE(p.date)=DATE(s.Date) AND s.Ticker='^GSPC'
    JOIN stock_prices n ON DATE(p.date)=DATE(n.Date) AND n.Ticker='^NSEI'
    ORDER BY p.date
    """

    df = pd.read_sql(query, conn).dropna()

    if len(date_range) == 2:
        df = df[(df["date"] >= str(date_range[0])) & (df["date"] <= str(date_range[1]))]

    col1, col2, col3, col4 = st.columns(4)

    col1.metric("Bitcoin Avg", round(df["bitcoin"].mean(), 2))
    col2.metric("Oil Avg", round(df["oil"].mean(), 2))
    col3.metric("S&P 500 Avg", round(df["sp500"].mean(), 2))
    col4.metric("NIFTY Avg", round(df["nifty"].mean(), 2))

    st.subheader("📋 Daily Market Snapshot")
    st.dataframe(df)

# =========================================================
#  PAGE 2: SQL RUNNER (ALL QUERIES)
# =========================================================
elif page == "Page 2: SQL Runner":

    st.title(" SQL Query Runner")

    queries = {

    # =====================================================
    # 🪙 CRYPTOCURRENCIES TABLE
    # =====================================================

    "1. Find the top 3 cryptocurrencies based on market capitalization":
    """
    SELECT name, market_cap
    FROM cryptocurrencies
    ORDER BY market_cap DESC
    LIMIT 3
    """,

    "2. List all cryptocurrencies where circulating supply exceeds 90% of total supply":
    """
    SELECT name,
      circulating_supply,
      total_supply,
       (circulating_supply * 100.0 / total_supply) AS percentage
        FROM cryptocurrencies
      WHERE circulating_supply > 0.9 * total_supply
     AND circulating_supply < total_supply

    """,

    "3. Get cryptocurrencies that are within 10% of their all-time high (ATH)":
    """
    SELECT name,current_price, ath
    FROM cryptocurrencies
    WHERE current_price >= 0.9 * ath
    """,

    "4. Find the average market cap rank of coins with trading volume above 1 billion USD":
    """
    SELECT AVG(market_cap_rank)
    FROM cryptocurrencies
    WHERE total_volume > 1000000000
    """,

    "5. Get the most recently updated cryptocurrency record":
    """
    SELECT name,date
    FROM cryptocurrencies
    ORDER BY date DESC
    LIMIT 1
    """,

    # =====================================================
    # 🪙 CRYPTO PRICES
    # =====================================================

    "6. Find the highest daily price of Bitcoin in the last 365 days":
    """
    SELECT MAX(price_usd)
    FROM crypto_prices
    WHERE coin_id='bitcoin'
    AND date >= DATE('now','-365 days')
    ORDER BY price_usd DESC
    LIMIT 1
    """,

    "7. Calculate the average daily price of Ethereum in the past 1 year":
    """
    SELECT AVG(price_usd)
    FROM crypto_prices
    WHERE coin_id='ethereum'
    AND date >= DATE('now','-365 days')
    """,

    "8. Show the daily price trend of Bitcoin in January 2026":
    """
    SELECT DISTINCT date, price_usd
     FROM crypto_prices
     WHERE coin_id = 'bitcoin'
      AND strftime('%Y-%m', date) = '2026-01'
      ORDER BY date;
    """,

    "9. Find the coin with the highest average price over the last 1 year":
    """
    SELECT coin_id, AVG(price_usd) AS avg_price
    FROM crypto_prices
    WHERE date >= DATE('now','-365 days')
    GROUP BY coin_id
    ORDER BY avg_price DESC
    LIMIT 1
    """,

    "10. Get the percentage change in Bitcoin price between Sep 2025 and Mar 2026":
    """
    SELECT
(
    (SELECT price_usd FROM crypto_prices
     WHERE coin_id='bitcoin'
     AND strftime('%Y-%m', date) = '2026-03'
     ORDER BY date DESC LIMIT 1)

    -

    (SELECT price_usd FROM crypto_prices
     WHERE coin_id='bitcoin'
     AND strftime('%Y-%m', date) = '2025-03'
     ORDER BY date DESC LIMIT 1)
)
* 100.0 /

(SELECT price_usd FROM crypto_prices
 WHERE coin_id='bitcoin'
 AND strftime('%Y-%m', date) = '2025-03'
 ORDER BY date DESC
 LIMIT 1)
 AS percent_change

    """,

    # =====================================================
    # 🛢️ OIL PRICES
    # =====================================================

    "11. Find the highest oil price in the last 5 years":
    """
    SELECT MAX(price)
    FROM oil_prices
    WHERE date >= DATE('now', '-5 years')
    LIMIT 1
    """,

    "12. Get the average oil price per year":
    """
    SELECT strftime('%Y', date) AS year, AVG(price)
    FROM oil_prices
    GROUP BY year
    """,

    "13. Show oil prices during COVID crash (March–April 2020)":
    """
    SELECT date,price
    FROM oil_prices
    WHERE date BETWEEN '2020-03-01' AND '2020-04-30'
    """,

    "14. Find the lowest oil price in the last 10 years":
    """
    SELECT MIN(price)
    FROM oil_prices
    WHERE date >= DATE('now','-10 years')
    ORDER BY price ASC
    LIMIT 1
    """,

    "15. Calculate yearly oil price volatility (max-min difference per year)":
    """
    SELECT strftime('%Y', date) AS year,
    MAX(price) - MIN(price) AS volatility
    FROM oil_prices
    GROUP BY year
    ORDER BY year
    """,

    # =====================================================
    # 📈 STOCK PRICES
    # =====================================================

    "16. Get all stock prices for given ticker":
    """
    SELECT *
FROM stock_prices
WHERE ticker IN ('^GSPC', '^IXIC', '^NSEI')
ORDER BY ticker,date;
      """,
    "17. Find the highest closing price for NASDAQ (^IXIC):":
    """
    SELECT *
    FROM stock_prices
    WHERE Ticker='^IXIC'
    ORDER BY Close DESC
    LIMIT 1
    """,

    "18. List top 5 days with highest price difference (High - Low) for S&P 500":
    """
    SELECT Date, (High - Low) AS price_diff
    FROM stock_prices
    WHERE Ticker='^GSPC'
    ORDER BY price_diff DESC
    LIMIT 5
    """,

    "19. Get monthly average closing price for each ticker":
    """
    SELECT Ticker, strftime('%Y-%m', Date)AS month,
    AVG(Close) AS avg_close
    FROM stock_prices
    GROUP BY Ticker, 2
    """,

    "20. Get average trading volume of NSEI in 2026":
    """
    SELECT AVG(Volume)
FROM stock_prices
WHERE Ticker = '^NSEI'
AND strftime('%Y', Date) = '2026'
    """,

    # =====================================================
    # 🔗 CROSS MARKET
    # =====================================================

    "21. Compare Bitcoin vs Oil average price in 2026":
    """
    SELECT
    DATE(p.date) as date,
    AVG(p.price_usd) AS bitcoin_price,
    AVG(s.Close) AS sp500
FROM crypto_prices p
INNER JOIN stock_prices s
ON DATE(p.date) = DATE(s.Date)
AND s.Ticker = '^GSPC'
WHERE p.coin_id = 'bitcoin'
AND strftime('%Y', p.date) = '2026'
GROUP BY DATE(p.date)
ORDER BY date;
    """,

    "22. Check if Bitcoin moves with S&P 500":
    """
    SELECT p.date, p.price_usd, s.Close
    FROM crypto_prices p
    JOIN stock_prices s ON DATE(p.date)=DATE(s.Date)
    WHERE p.coin_id='bitcoin'
    AND s.Ticker='^GSPC'
    """,

    "23. Compare Ethereum and NASDAQ daily prices for 2026":
    """
    SELECT
    p.date,
    p.price_usd AS ethereum_price,
    s.Close AS nasdaq
FROM crypto_prices p
INNER JOIN stock_prices s
ON DATE(p.date) = DATE(s.Date)
AND s.Ticker = '^IXIC'
WHERE p.coin_id = 'ethereum'
AND strftime('%Y', p.date) = '2026'
ORDER BY p.date
    """,

    "24. Find days when oil price spiked and compare with Bitcoin":
    """
    SELECT
    DATE(o.Date) as Date,
    AVG(o.Price) AS oil_price,
    p.price_usd AS bitcoin_price
FROM oil_prices o
INNER JOIN crypto_prices p
ON DATE(o.Date) = DATE(p.date)
WHERE p.coin_id = 'bitcoin'
GROUP BY DATE(o.Date), p.price_usd
ORDER BY Date
    """,

    "25. Compare top cryptocurrencies with NIFTY index":
    """
    SELECT
    p.date,
    p.coin_id,
    p.price_usd,
    s.Close AS nifty
FROM crypto_prices p
INNER JOIN stock_prices s
ON DATE(p.date) = DATE(s.Date)
WHERE p.coin_id IN ('bitcoin','ethereum','tether')
AND s.Ticker = '^NSEI'
ORDER BY p.date

    """,
    "26.Compare stock prices(^GSPC) with crude oil prices on the same dates":
    """
    SELECT
    s.Date,
    s.Close AS sp500,
    o.Price AS oil_price
FROM stock_prices s
INNER JOIN oil_prices o
ON DATE(s.Date) = DATE(o.Date)
WHERE s.Ticker = '^GSPC'
ORDER BY s.Date;

""",
"27.Correlate Bitcoin closing price with crude oil closing price (same date)":
     """
     SELECT
    p.date,
    p.price_usd AS bitcoin,
    o.Price AS oil
FROM crypto_prices p
INNER JOIN oil_prices o
ON DATE(p.date) = DATE(o.Date)
WHERE p.coin_id = 'bitcoin'
ORDER BY p.date;
""",
 "28.Compare NASDAQ (^IXIC) with Ethereum price trends":
   """
SELECT
    p.date,
    AVG(p.price_usd) AS ethereum,
    AVG(s.close) AS nasdaq
FROM crypto_prices p
JOIN stock_prices s
ON DATE(p.date) = DATE(s.date)
WHERE p.coin_id = 'ethereum'
AND s.ticker = '^IXIC'
GROUP BY p.date
ORDER BY p.date
""",
    "29.Join top 3 crypto coins with stock indices for 2026":
    """
SELECT
    DATE(p.date) AS date,
    p.coin_id,
    AVG(p.price_usd) AS price_usd,
    MAX(CASE WHEN s.Ticker='^NSEI' THEN s.Close END) AS nsei,
    MAX(CASE WHEN s.Ticker='^GSPC' THEN s.Close END) AS sp500,
    MAX(CASE WHEN s.Ticker='^IXIC' THEN s.Close END) AS nasdaq
FROM crypto_prices p
INNER JOIN stock_prices s
ON DATE(p.date) = DATE(s.Date)
WHERE p.coin_id IN ('bitcoin','ethereum','tether')
AND strftime('%Y', p.date) = '2026'
GROUP BY DATE(p.date), p.coin_id
ORDER BY date
""",
    "30. Multi-market comparison (crypto, oil, stocks)":

    """
SELECT
    p.date,
    p.price_usd AS bitcoin,
    s.close AS sp500,
    o.price AS oil
FROM crypto_prices p
JOIN (
    SELECT date, close
    FROM stock_prices
    WHERE ticker = '^GSPC'
) s ON DATE(p.date) = DATE(s.date)
JOIN (
    SELECT date, price
    FROM oil_prices
) o ON DATE(p.date) = DATE(o.date)
WHERE p.coin_id = 'bitcoin'
GROUP BY p.date
ORDER BY p.date
"""
    }

    selected_query = st.selectbox("Select Query", list(queries.keys()))

    if st.button("Run Query"):
        df = pd.read_sql(queries[selected_query], conn).dropna()
        st.success("✅ Query executed successfully")
        st.dataframe(df)

# =========================================================
# 📈 PAGE 3: CRYPTO ANALYSIS
# =========================================================
elif page == "Page 3: Crypto Analysis":

    st.title("📈 Crypto Analysis")

    coin = st.selectbox("Select Coin", ["bitcoin", "ethereum", "tether"])
    date_range = st.date_input("Select Date Range", [])

    query = f"""
    SELECT date, price_usd
    FROM crypto_prices
    WHERE coin_id='{coin}'
    ORDER BY date
    """

    df = pd.read_sql(query, conn).dropna()

    if len(date_range) == 2:
        df = df[(df["date"] >= str(date_range[0])) & (df["date"] <= str(date_range[1]))]

    st.subheader("📊 Price Trend")
    st.line_chart(df.set_index("date"))

    st.subheader("📋 Data Table")
    st.dataframe(df)

# -------------------------------
# CLOSE CONNECTION
# -------------------------------
conn.close()

Overwriting app.py


In [ ]:
!streamlit run /content/app.py &>/content/logs.txt &

In [ ]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://needle-alpha-buys-sleep.trycloudflare.com
